
# Rastreando a origem do aumento de inadimplência, fotografia por fotografia

Prova de conceito com **dados simulados** para ilustrar um framework de
diagnóstico: separar, a cada fotografia mensal, três efeitos que se
misturam num número agregado de inadimplência (idade/MOB, safra/coorte,
época/período), e detectar deterioração sustentada com um critério
estatístico explícito em vez de leitura visual.

Nesta versão, o notebook reflete três decisões de desenho fechadas nas
últimas rodadas:

1. **A base de origem é truncada à esquerda.** Contratos originados antes
   do início da janela de carregamento só aparecem a partir do MOB
   correspondente a essa data de início — sem histórico anterior
   recuperável. Isso é diferente de censura à direita (safra jovem que
   ainda não viveu MOB alto): aqui, o pedaço que falta nunca vai aparecer.
2. **A agregação pesada roda em Spark; a modelagem estatística continua em
   pandas** — não existe equivalente direto de GLM com fórmula arbitrária
   em Spark, e a agregação já reduz o volume antes de chegar em pandas.
3. **A macro entra como opção liga/desliga** (`USA_MACRO`), indexada por
   `mes_referencia` (aplicada a toda safra viva naquele mês, não fixada
   por safra) — porque só assim ela cumpre o papel de ancorar o efeito de
   época e resolver a colinearidade idade-período-safra.


In [1]:

import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_rows", 30)



## 1. Simulação: tabela bruta no grão de contrato-mês, com estoque herdado

Diferente da versão anterior (que já nascia agregada em células
safra×MOB), agora simulo primeiro uma tabela **granular** — uma linha por
contrato por mês — porque é esse o formato realista de uma fonte
regulatória empilhada mensalmente, e é isso que uma camada Spark de
verdade seria chamada para agregar.

Cenário:

- 24 safras **dentro da janela de observação** (mês 1 a 24), 800 contratos
  cada, com a safra ruim (9, 10, 11) e o choque de época a partir do mês
  15, exatamente como antes.
- **+ 12 safras anteriores ao início da janela** (safra -11 a 0), também
  800 contratos cada na origem — simuladas com o mesmo processo, inclusive
  nos meses anteriores ao início da janela, para que o atrito natural
  (entrada em atraso) já tenha acontecido antes da janela abrir. Só então
  a tabela bruta passa a registrar essas linhas a partir do mês 1 — isso
  gera, de forma estrutural (não arbitrária), a mesma população
  sobrevivente-viesada que uma fonte truncada de verdade teria: quem já
  tinha saído antes da janela abrir nunca entra na base.
- Cada contrato para de gerar linha assim que entra em atraso grave
  (estado absorvente, consistente com a definição de fluxo de entrada).


In [2]:

N_SAFRAS = 24
N_POR_SAFRA = 800
PEAK_MOB = 7
BASE_RATE = 0.018
SAFRAS_RUINS = {9, 10, 11}
EFEITO_COHORT_RUIM = 0.5
INICIO_CHOQUE_MACRO = 15
PRE_JANELA_SAFRAS = 12          # safras -11 a 0, existentes antes da janela abrir
SAFRA_MIN = 1 - PRE_JANELA_SAFRAS

def hazard_base(mob):
    '''Curva de risco por idade (MOB), formato de sino com pico em PEAK_MOB.'''
    return BASE_RATE * (mob / PEAK_MOB) * np.exp(1 - mob / PEAK_MOB)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def construir_macro_verdadeiro(n_meses, inicio_choque):
    '''So definido para t=1..n_meses (dentro da janela) -- antes da janela, sem choque.'''
    m = np.zeros(n_meses + 1)
    for t in range(1, n_meses + 1):
        if t >= inicio_choque:
            m[t] = min(0.9, (t - (inicio_choque - 1)) * 0.9 / 10)
    return m

def gerar_tabela_bruta_contratual(rng):
    '''Simula contrato a contrato, mes a mes (inclusive antes da janela abrir),
    mas so registra na tabela final as linhas com mes_referencia >= 1 --
    o que corresponde ao que uma fonte truncada de verdade conseguiria mostrar.'''
    macro_verdadeiro = construir_macro_verdadeiro(N_SAFRAS, INICIO_CHOQUE_MACRO)
    estado_ativo = {s: np.ones(N_POR_SAFRA, dtype=bool) for s in range(SAFRA_MIN, N_SAFRAS + 1)}
    partes = []
    for t in range(SAFRA_MIN, N_SAFRAS + 1):
        for s in range(SAFRA_MIN, min(t, N_SAFRAS) + 1):
            mob = t - s + 1
            if mob < 1:
                continue
            ativos = estado_ativo[s]
            n_em_risco = int(ativos.sum())
            if n_em_risco == 0:
                continue
            cohort_efeito = EFEITO_COHORT_RUIM if s in SAFRAS_RUINS else 0.0
            macro_t = macro_verdadeiro[t] if 1 <= t <= N_SAFRAS else 0.0
            h_base = hazard_base(mob)
            logit_base = np.log(h_base / (1 - h_base))
            p = sigmoid(logit_base + cohort_efeito + macro_t)
            sorteio = rng.random(n_em_risco) < p

            if t >= 1:  # so o que a janela de observacao conseguiria enxergar
                idx = np.where(ativos)[0]
                partes.append(pd.DataFrame({
                    "id_contrato": [f"S{s}_C{i}" for i in idx],
                    "safra": s, "mes_referencia": t, "mob": mob,
                    "flag_entrante_90mais": sorteio.astype(int),
                }))
            idx_ativos = np.where(ativos)[0]
            estado_ativo[s][idx_ativos[sorteio]] = False

    tabela = pd.concat(partes, ignore_index=True)
    tabela["veio_truncado"] = (tabela["safra"] < 1).astype(int)
    return tabela

rng_principal = np.random.default_rng(42)
t0 = time.time()
tabela_bruta = gerar_tabela_bruta_contratual(rng_principal)
print(f"Tabela bruta gerada em {time.time()-t0:.2f}s -- {len(tabela_bruta)} linhas (grao contrato-mes)")
tabela_bruta.head()


Tabela bruta gerada em 0.40s -- 392435 linhas (grao contrato-mes)


,id_contrato,safra,mes_referencia,mob,flag_entrante_90mais,veio_truncado
0,S-11_C0,-11,1,13,0,1
1,S-11_C1,-11,1,13,0,1
2,S-11_C2,-11,1,13,0,1
3,S-11_C3,-11,1,13,0,1
4,S-11_C4,-11,1,13,0,1



### Checagem do truncamento antes de seguir

Para uma safra pré-janela, o primeiro MOB observado tem que ser **maior
que 1** (o começo da vida dela nunca aparece); para uma safra dentro da
janela, o primeiro MOB observado tem que ser **exatamente 1**. Se isso não
bater, o truncamento não foi implementado direito.


In [3]:

print("MOB minimo por safra pre-janela (esperado: sempre > 1):")
print(tabela_bruta.loc[tabela_bruta["safra"] < 1].groupby("safra")["mob"].min())
print("\nMOB minimo por safra dentro da janela, 5 primeiras (esperado: sempre = 1):")
print(tabela_bruta.loc[tabela_bruta["safra"] >= 1].groupby("safra")["mob"].min().head())

pop_inicial = tabela_bruta.loc[tabela_bruta["mes_referencia"] == 1].groupby("veio_truncado")["id_contrato"].count()
print(f"\nPopulacao no mes 1 -- nova (veio_truncado=0): {pop_inicial.get(0,0)} "
      f"| herdada (veio_truncado=1): {pop_inicial.get(1,0)} de um estoque teorico de "
      f"{PRE_JANELA_SAFRAS * N_POR_SAFRA} (a diferenca ja saiu por atraso grave antes da janela abrir)")


MOB minimo por safra pre-janela (esperado: sempre > 1):
safra
-11    13
-10    12
-9     11
-8     10
-7      9
-6      8
-5      7
-4      6
-3      5
-2      4
-1      3
 0      2
Name: mob, dtype: int64

MOB minimo por safra dentro da janela, 5 primeiras (esperado: sempre = 1):
safra
1    1
2    1
3    1
4    1
5    1
Name: mob, dtype: int64

Populacao no mes 1 -- nova (veio_truncado=0): 800 | herdada (veio_truncado=1): 8738 de um estoque teorico de 9600 (a diferenca ja saiu por atraso grave antes da janela abrir)



## 2. Camada de ingestão: agregação em Spark, modelagem em pandas

A tabela bruta acima é o formato que uma fonte real entregaria (grão
contrato-mês). A agregação para o grão safra×MOB×mês — a única etapa
computacionalmente pesada, já que reduz potencialmente milhões de linhas a
algumas centenas de células — é o que rodaria em Spark numa carteira real.
Aqui, simulo essa fronteira de verdade: gero um `DataFrame` Spark a partir
da tabela bruta, agrego nele, e só then trago o resultado (já pequeno) de
volta para pandas.


In [4]:

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("carteira_agregacao").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

t1 = time.time()
df_spark_bruto = spark.createDataFrame(tabela_bruta)

df_spark_agregado = (
    df_spark_bruto
    .groupBy("safra", "mob", "mes_referencia")
    .agg(
        F.count("*").alias("populacao_risco"),
        F.sum("flag_entrante_90mais").alias("entrantes_90mais"),
    )
)

painel = df_spark_agregado.toPandas().rename(columns={"mes_referencia": "mes_calendario"})
print(f"Agregacao Spark -> pandas em {time.time()-t1:.2f}s -- {len(painel)} celulas (era {len(tabela_bruta)} linhas)")

# checagem cruzada: o total agregado pelo Spark tem que bater com o total direto na tabela bruta
total_spark = painel["entrantes_90mais"].sum()
total_bruto = tabela_bruta["flag_entrante_90mais"].sum()
assert total_spark == total_bruto, "Divergencia entre agregado Spark e tabela bruta -- nao seguir sem investigar"
print(f"Checagem cruzada ok: {total_spark} entrantes em ambos os calculos")

spark.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 03:35:04 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/28 03:35:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/08/28 03:35:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Agregacao Spark -> pandas em 20.09s -- 588 celulas (era 392435 linhas)
Checagem cruzada ok: 6525 entrantes em ambos os calculos


In [5]:

# do ponto em diante, tudo em pandas -- macro, mob_bin e veio_truncado sao
# transformacoes leves, sem justificativa para rodar em Spark
painel["veio_truncado"] = (painel["safra"] < 1).astype(int)

MOB_MAXIMO = N_SAFRAS - SAFRA_MIN + 1  # maior MOB possivel, dado o estoque herdado
bins_mob = list(range(0, MOB_MAXIMO + 3, 3))
labels_mob = [f"{i+1}-{i+3}" for i in bins_mob[:-1]]
painel["mob_bin"] = pd.cut(painel["mob"], bins=bins_mob, labels=labels_mob)

macro_verdadeiro = construir_macro_verdadeiro(N_SAFRAS, INICIO_CHOQUE_MACRO)
macro_reportado_serie = np.zeros(N_SAFRAS + 1)
for t in range(1, N_SAFRAS + 1):
    t_ref = max(1, t - 2)  # defasagem de divulgacao de 2 meses
    macro_reportado_serie[t] = macro_verdadeiro[t_ref] + rng_principal.normal(0, 0.02)
painel["macro_reportado"] = painel["mes_calendario"].map(lambda t: macro_reportado_serie[t])

painel["taxa_entrada"] = painel["entrantes_90mais"] / painel["populacao_risco"]
painel.sort_values(["safra", "mob"]).head()


,safra,mob,mes_calendario,populacao_risco,entrantes_90mais,veio_truncado,mob_bin,macro_reportado,taxa_entrada
409,-11,13,1,671,13,1,13-15,0.033067,0.019374
160,-11,14,2,658,6,1,13-15,-0.056209,0.009119
119,-11,15,3,652,8,1,13-15,-0.010802,0.012270
540,-11,16,4,644,7,1,16-18,0.001998,0.010870
438,-11,17,5,637,7,1,16-18,0.005161,0.010989


### Checagem rápida: a inadimplência agregada está mesmo subindo?

In [6]:

total_mensal = painel.groupby("mes_calendario")["entrantes_90mais"].sum().reset_index()

fig_check = go.Figure()
fig_check.add_trace(go.Scatter(
    x=total_mensal["mes_calendario"], y=total_mensal["entrantes_90mais"],
    mode="lines+markers", name="Novos entrantes em 90+ / mes",
    line=dict(color="#B23A48", width=2),
))
fig_check.add_vline(x=INICIO_CHOQUE_MACRO, line_dash="dot", line_color="gray",
                     annotation_text="inicio do choque de epoca (nao visivel na fonte de dados ainda)")
fig_check.update_layout(
    title="Total de novos entrantes em atraso grave por mes de referencia (fotografia)",
    xaxis_title="Mes de referencia (fotografia)", yaxis_title="Novos entrantes",
    template="plotly_white", width=1400, height=420,
)
fig_check.show()



## 3. Modelo de referência: idade + (macro opcional) + truncamento

`USA_MACRO` liga ou desliga o termo de macro na fórmula -- útil enquanto
a base macro por safra/época ainda não está disponível na prática.
**Consequência de desligar**: o efeito de período deixa de ter âncora
nenhuma -- qualquer variação de época cai inteira no resíduo que alimenta
o teste de Page (Seção 6). Isso não quebra a detecção, mas o teste deixa
de distinguir "choque econômico conhecido" de "anomalia sem explicação
disponível".

`veio_truncado` entra como covariável -- testa formalmente se a população
herdada (sobrevivente do estoque pré-janela) se comporta, sistematicamente,
diferente do resto, depois de já controlado por idade e época. Ele **não**
entra no modelo de efeito de safra (Seção 7): lá, cada safra já tem seu
próprio efeito fixo, e como toda safra pré-janela tem `veio_truncado=1`
por definição, incluir os dois juntos seria colinear.


In [7]:

USA_MACRO = True

def formula_modelo_a():
    base = "entrantes_90mais ~ C(mob_bin) + veio_truncado"
    return base + " + macro_reportado" if USA_MACRO else base

FORMULA_MODELO_A = formula_modelo_a()
print("Formula em uso:", FORMULA_MODELO_A)

def ajustar_modelo_a(dados_treino):
    return smf.glm(
        formula=FORMULA_MODELO_A, data=dados_treino,
        family=sm.families.Poisson(), offset=np.log(dados_treino["populacao_risco"]),
    ).fit()

modelo_referencia_completo = ajustar_modelo_a(painel)
print(modelo_referencia_completo.summary())


Formula em uso: entrantes_90mais ~ C(mob_bin) + veio_truncado + macro_reportado
                 Generalized Linear Model Regression Results                  
Dep. Variable:       entrantes_90mais   No. Observations:                  588
Model:                            GLM   Df Residuals:                      574
Model Family:                 Poisson   Df Model:                           13
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1577.1
Date:                Fri, 28 Aug 2026   Deviance:                       795.50
Time:                        03:35:34   Pearson chi2:                     784.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.9455
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------


## 4. Backtest um passo à frente, e o CUSUM ingênuo (para contraste)

Mesma lógica de sempre: ajustar só com dados até o mês anterior, prever o
mês novo, acumular o resíduo padronizado sem critério de decisão -- o
problema que a Seção 5 resolve.

`MES_INICIO_BACKTEST` continua conservador (mês 6) mesmo com o estoque
herdado cobrindo MOB alto desde o mês 1: a cobertura de idade melhorou,
mas o número de **safras novas dentro da janela** (o que dá variação para
separar época de coorte-safra-nova) ainda é pouco nos primeiros meses --
então a calibração inicial continua necessária, só que por um motivo
diferente do notebook anterior.


In [8]:

MES_INICIO_BACKTEST = 6
resultados_backtest = []
for t_foto in range(MES_INICIO_BACKTEST, N_SAFRAS + 1):
    treino = painel[painel["mes_calendario"] <= t_foto - 1].copy()
    novo = painel[painel["mes_calendario"] == t_foto].copy()
    if novo.empty:
        continue
    try:
        modelo_t = ajustar_modelo_a(treino)
        esperado = modelo_t.predict(novo, offset=np.log(novo["populacao_risco"]))
    except Exception as erro:
        print(f"Fotografia {t_foto}: ajuste falhou ({erro}) -- pulando")
        continue

    n_celulas = len(novo)
    residuo_pearson = (
        (novo["entrantes_90mais"].values - esperado.values) / np.sqrt(np.maximum(esperado.values, 0.5))
    ).sum()
    resultados_backtest.append(dict(
        mes_calendario=t_foto, observado=novo["entrantes_90mais"].sum(),
        esperado=esperado.sum(), residuo_pearson=residuo_pearson, n_celulas=n_celulas,
    ))

backtest_df = pd.DataFrame(resultados_backtest)
backtest_df["z"] = backtest_df["residuo_pearson"] / np.sqrt(backtest_df["n_celulas"])
backtest_df["cusum_ingenuo"] = backtest_df["residuo_pearson"].cumsum()
backtest_df


,mes_calendario,observado,esperado,residuo_pearson,n_celulas,z,cusum_ingenuo
0,6,181,173.136153,3.233436,18,0.762128,3.233436
1,7,224,186.292238,11.728062,19,2.690602,14.961498
2,8,184,192.906012,-1.699670,20,-0.380058,13.261828
3,9,194,198.691732,-2.336370,21,-0.509838,10.925458
4,10,205,207.842739,-2.268359,22,-0.483616,8.657099
5,11,192,211.980990,-6.975923,23,-1.454580,1.681177
6,12,204,209.416817,-1.375407,24,-0.280754,0.305769
7,13,218,206.766133,2.229287,25,0.445857,2.535057
8,14,257,214.143851,17.215408,26,3.376219,19.750465
9,15,267,224.631103,13.235007,27,2.547078,32.985472



## 5. Teste de Page: soma cumulativa com regra de decisão explícita

$$C^+_t = \max(0,\; C^+_{t-1} + z_t - k) \qquad C^-_t = \max(0,\; C^-_{t-1} - z_t - k)$$

`k` (folga) impede que ruído comum se acumule; `h` (intervalo de decisão)
é o limite que `C+`/`C-` precisam cruzar para virar alarme formal, e é
calibrado por simulação do cenário nulo (sem safra ruim, sem choque de
época), não por tabela genérica -- os dois problemas do CUSUM ingênuo.

**Limite importante desta versão**: a calibração abaixo usa um simulador
mais simples (`simular_painel_nulo`, sem estoque herdado nem truncamento)
por custo computacional -- rodar 300 réplicas do pipeline completo
(contrato a contrato + Spark) seria caro demais para este notebook. Isso
significa que o `h` calibrado aqui é uma referência aproximada, calibrada
contra um processo nulo ligeiramente mais simples que o processo real que
gera `backtest_df`. Para uso em produção, valeria a pena revisar se isso
importa (rodando algumas dezenas de réplicas completas e comparando).


In [9]:

def calcular_cusum_page(serie_z, k):
    c_mais, c_menos = np.zeros(len(serie_z)), np.zeros(len(serie_z))
    for i, z in enumerate(serie_z):
        ant_mais = c_mais[i - 1] if i > 0 else 0.0
        ant_menos = c_menos[i - 1] if i > 0 else 0.0
        c_mais[i] = max(0.0, ant_mais + z - k)
        c_menos[i] = max(0.0, ant_menos - z - k)
    return c_mais, c_menos

K_REFERENCIA = 0.5

def simular_painel_nulo(rng):
    '''Versao leve (sem estoque herdado) so para a calibracao de h -- ver ressalva acima.'''
    registros = []
    estado_ativo = {s: np.ones(N_POR_SAFRA, dtype=bool) for s in range(1, N_SAFRAS + 1)}
    for t in range(1, N_SAFRAS + 1):
        for s in range(1, t + 1):
            mob = t - s + 1
            ativos = estado_ativo[s]
            n = int(ativos.sum())
            if n == 0:
                continue
            h_base = hazard_base(mob)
            p = sigmoid(np.log(h_base / (1 - h_base)))  # sem cohort, sem macro -- cenario nulo
            sorteio = rng.random(n) < p
            idx = np.where(ativos)[0]
            estado_ativo[s][idx[sorteio]] = False
            registros.append(dict(mes_calendario=t, safra=s, mob=mob,
                                   entrantes_90mais=int(sorteio.sum()), populacao_risco=n))
    df = pd.DataFrame(registros)
    df["veio_truncado"] = 0
    bins_mob_nulo = list(range(0, N_SAFRAS + 3, 3))
    labels_mob_nulo = [f"{i+1}-{i+3}" for i in bins_mob_nulo[:-1]]
    df["mob_bin"] = pd.cut(df["mob"], bins=bins_mob_nulo, labels=labels_mob_nulo)
    df["macro_reportado"] = 0.0
    return df

N_REPLICACOES_NULAS = 300
PERCENTIL_H = 95
maximos_c_mais_nulo = []
for i in range(N_REPLICACOES_NULAS):
    rng_i = np.random.default_rng(1000 + i)
    painel_nulo = simular_painel_nulo(rng_i)
    linhas = []
    for t_foto in range(MES_INICIO_BACKTEST, N_SAFRAS + 1):
        treino = painel_nulo[painel_nulo["mes_calendario"] <= t_foto - 1]
        novo = painel_nulo[painel_nulo["mes_calendario"] == t_foto]
        if novo.empty:
            continue
        try:
            modelo_i = ajustar_modelo_a(treino)
            esperado = modelo_i.predict(novo, offset=np.log(novo["populacao_risco"]))
        except Exception:
            continue
        n_celulas = len(novo)
        residuo = ((novo["entrantes_90mais"].values - esperado.values)
                   / np.sqrt(np.maximum(esperado.values, 0.5))).sum()
        linhas.append(residuo / np.sqrt(n_celulas))
    if len(linhas) < 3:
        continue
    c_mais_i, _ = calcular_cusum_page(np.array(linhas), K_REFERENCIA)
    maximos_c_mais_nulo.append(c_mais_i.max())

maximos_c_mais_nulo = np.array(maximos_c_mais_nulo)
H_EMPIRICO = float(np.percentile(maximos_c_mais_nulo, PERCENTIL_H))
print(f"Replicacoes nulas validas: {len(maximos_c_mais_nulo)} de {N_REPLICACOES_NULAS}")
print(f"h calibrado (percentil {PERCENTIL_H} do maximo sob H0): {H_EMPIRICO:.2f}")


Replicacoes nulas validas: 300 de 300
h calibrado (percentil 95 do maximo sob H0): 6.20



## 6. Aplicando o teste de Page ao cenário com choque


In [10]:

c_mais, c_menos = calcular_cusum_page(backtest_df["z"].values, K_REFERENCIA)
backtest_df["c_mais"] = c_mais
backtest_df["c_menos"] = c_menos

meses_alarme_alta = backtest_df.loc[backtest_df["c_mais"] >= H_EMPIRICO, "mes_calendario"]
meses_alarme_queda = backtest_df.loc[backtest_df["c_menos"] >= H_EMPIRICO, "mes_calendario"]
print(f"Primeiro alarme de piora (C+ >= {H_EMPIRICO:.2f}): "
      f"{int(meses_alarme_alta.min()) if not meses_alarme_alta.empty else 'nenhum'}")
print(f"Primeiro alarme de melhora (C- >= {H_EMPIRICO:.2f}): "
      f"{int(meses_alarme_queda.min()) if not meses_alarme_queda.empty else 'nenhum'} "
      f"-- lembrar de checar se e melhora real ou recalibracao do modelo antes de aceitar")

backtest_df[["mes_calendario", "z", "c_mais", "c_menos"]]


Primeiro alarme de piora (C+ >= 6.20): 16
Primeiro alarme de melhora (C- >= 6.20): 24 -- lembrar de checar se e melhora real ou recalibracao do modelo antes de aceitar


,mes_calendario,z,c_mais,c_menos
0,6,0.762128,0.262128,0.000000
1,7,2.690602,2.452730,0.000000
2,8,-0.380058,1.572672,0.000000
3,9,-0.509838,0.562835,0.009838
4,10,-0.483616,0.000000,0.000000
5,11,-1.454580,0.000000,0.954580
6,12,-0.280754,0.000000,0.735334
7,13,0.445857,0.000000,0.000000
8,14,3.376219,2.876219,0.000000
9,15,2.547078,4.923298,0.000000



### Visualização evoluindo fotografia por fotografia

O mapa agora inclui as safras pré-janela (índices negativos), separadas da
janela de observação por uma linha pontilhada. Repare que, ao contrário
das safras dentro da janela (que preenchem o triângulo MOB=1,2,3... aos
poucos), as safras pré-janela **nunca preenchem os MOBs baixos** -- ficam
permanentemente em branco à esquerda, porque esse pedaço do histórico
delas nunca existiu na fonte. É a diferença visual entre truncamento
(buraco permanente) e censura (buraco temporário).


In [11]:

safras_ordenadas = list(range(SAFRA_MIN, N_SAFRAS + 1))
mobs_ordenados = list(range(1, MOB_MAXIMO + 1))

def matriz_revelada_ate(t_foto):
    matriz = np.full((len(safras_ordenadas), len(mobs_ordenados)), np.nan)
    visivel = painel[painel["mes_calendario"] <= t_foto]
    for _, linha in visivel.iterrows():
        i = int(linha["safra"]) - SAFRA_MIN
        j = int(linha["mob"]) - 1
        matriz[i, j] = linha["taxa_entrada"]
    return matriz

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Mapa safra x idade (MOB) -- taxa de entrada em 90+",
                     f"Teste de Page: C+ (piora) e C- (melhora), limite h={H_EMPIRICO:.1f}"),
    column_widths=[0.5, 0.5],
)

z0 = matriz_revelada_ate(1)
fig.add_trace(go.Heatmap(
    z=z0, x=mobs_ordenados, y=safras_ordenadas, colorscale="Reds",
    zmin=0, zmax=painel["taxa_entrada"].quantile(0.98), colorbar=dict(title="taxa", x=0.46),
), row=1, col=1)
fig.add_hline(y=0.5, line_dash="dot", line_color="gray",
              annotation_text="inicio da janela de observacao", row=1, col=1)

fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="C+ (piora sustentada)",
                          line=dict(color="#B23A48", width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", name="C- (melhora sustentada)",
                          line=dict(color="#2A6F97", width=2)), row=1, col=2)
fig.add_hline(y=H_EMPIRICO, line_dash="dash", line_color="black",
              annotation_text=f"h = {H_EMPIRICO:.1f}", row=1, col=2)

frames = []
for t_foto in range(1, N_SAFRAS + 1):
    z_t = matriz_revelada_ate(t_foto)
    sub = backtest_df[backtest_df["mes_calendario"] <= t_foto]
    frames.append(go.Frame(
        name=str(t_foto),
        data=[
            go.Heatmap(z=z_t, x=mobs_ordenados, y=safras_ordenadas, colorscale="Reds",
                       zmin=0, zmax=painel["taxa_entrada"].quantile(0.98)),
            go.Scatter(x=sub["mes_calendario"], y=sub["c_mais"], mode="lines+markers",
                      line=dict(color="#B23A48", width=2)),
            go.Scatter(x=sub["mes_calendario"], y=sub["c_menos"], mode="lines+markers",
                      line=dict(color="#2A6F97", width=2)),
        ],
    ))
fig.frames = frames

steps = [
    dict(method="animate", label=str(t_foto),
         args=[[str(t_foto)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                    transition=dict(duration=0))])
    for t_foto in range(1, N_SAFRAS + 1)
]

fig.update_layout(
    width=1400, height=620,
    title="Evolucao fotografia por fotografia (arraste o controle abaixo)",
    xaxis_title="MOB (idade do contrato)", yaxis_title="Safra (mes de originacao; < 1 = pre-janela)",
    xaxis2_title="Fotografia (mes de referencia)", yaxis2_title="Estatistica de Page",
    template="plotly_white",
    sliders=[dict(active=0, currentvalue=dict(prefix="Fotografia (mes): "), steps=steps)],
    updatemenus=[dict(type="buttons", showactive=False, y=1.15, x=1.05,
                       buttons=[dict(label="Reproduzir", method="animate",
                                     args=[None, dict(frame=dict(duration=350, redraw=True),
                                                       fromcurrent=True)])])],
)
fig.show()



## 7. Isolando o efeito de safra (com alerta de credibilidade)

Ajuste com todo o histórico, agora incluindo as safras pré-janela como
dado adicional para as faixas de MOB alto (mais informação para o
`mob_bin`, o que ajuda a estimar melhor a curva de idade mesmo cedo no
histórico) -- mas o **ranking de efeito de safra exibido abaixo fica
restrito às safras dentro da janela** (1 a 24): são as únicas em que faz
sentido falar de "qualidade de subscrição" de forma acionável -- o
back-book herdado é o que é, não dá para agir retroativamente sobre ele.


In [12]:

FORMULA_MODELO_B = "entrantes_90mais ~ C(mob_bin) + macro_reportado + C(safra)" if USA_MACRO \
    else "entrantes_90mais ~ C(mob_bin) + C(safra)"

modelo_b = smf.glm(
    formula=FORMULA_MODELO_B, data=painel,
    family=sm.families.Poisson(), offset=np.log(painel["populacao_risco"]),
).fit()

safras_janela = list(range(1, N_SAFRAS + 1))
meses_observados_por_safra = painel.groupby("safra")["mob"].count()
efeitos = []
for s in safras_janela:
    nome_param = f"C(safra)[T.{s}]"
    if nome_param in modelo_b.params.index:
        coef, erro_padrao = modelo_b.params[nome_param], modelo_b.bse[nome_param]
    else:
        coef, erro_padrao = 0.0, 0.0
    efeitos.append(dict(safra=s, efeito=coef, erro_padrao=erro_padrao,
                         meses_observados=int(meses_observados_por_safra.get(s, 0))))

efeitos_df = pd.DataFrame(efeitos)
LIMIAR_CREDIVEL = 6
efeitos_df["credivel"] = efeitos_df["meses_observados"] >= LIMIAR_CREDIVEL
cores = np.where(efeitos_df["credivel"], "#B23A48", "#D9B8BC")

fig_safra = go.Figure()
fig_safra.add_trace(go.Bar(
    x=efeitos_df["safra"], y=efeitos_df["efeito"],
    error_y=dict(type="data", array=1.96 * efeitos_df["erro_padrao"], visible=True),
    marker_color=cores,
    text=[f"{m} meses" for m in efeitos_df["meses_observados"]],
    hovertemplate="Safra %{x}<br>Efeito: %{y:.3f}<br>%{text}<extra></extra>",
))
fig_safra.add_hline(y=0, line_dash="dot", line_color="gray")
fig_safra.update_layout(
    title=(f"Efeito de safra (dentro da janela) apos controlar idade e epoca -- tom claro = menos de "
           f"{LIMIAR_CREDIVEL} meses observados"),
    xaxis_title="Safra (mes de originacao)", yaxis_title="Efeito estimado (log-odds relativo)",
    template="plotly_white", width=1400, height=450,
)
fig_safra.show()

print("Safras com maior efeito estimado (entre as confiaveis):")
print(efeitos_df[efeitos_df["credivel"]].sort_values("efeito", ascending=False).head(6)
      [["safra", "efeito", "meses_observados"]])


Safras com maior efeito estimado (entre as confiaveis):
    safra    efeito  meses_observados
8       9  0.686205                16
10     11  0.684513                14
9      10  0.528432                15
17     18  0.459500                 7
13     14  0.319665                11
14     15  0.312712                10



## Leitura dos resultados e limites do exercício

Cenário injetado: safras 9, 10, 11 (dentro da janela) estruturalmente
piores; choque de época a partir do mês 15, visível ao modelo só com 2
meses de atraso; estoque pré-janela de 12 safras sofrendo o atrito normal
antes da janela abrir. Confira se o teste de Page aponta piora em torno
dos meses esperados e se a Seção 7 recupera as safras 9-11.

Limites que continuam valendo com dado real:

- **A calibração do `h` (Seção 5) usa um processo nulo mais simples que o
  processo real** (sem estoque herdado) -- ver a ressalva já registrada
  ali. Não é um erro escondido, é uma simplificação deliberada por custo,
  documentada para não ser esquecida.
- **`veio_truncado` e `C(safra)` não convivem na mesma fórmula** por
  colinearidade -- cada um responde uma pergunta diferente (Seção 3 vs.
  Seção 7), não são intercambiáveis.
- **A simulação não modela quitação antecipada**, só entrada em atraso
  como saída da população em risco. Numa carteira real, parte do estoque
  pré-janela também sairia por pagamento total antes da janela abrir --
  isso tornaria o viés de sobrevivência descrito na Seção 1 ainda mais
  forte do que o que este notebook reproduz.
- **`C-` cruzando `h` não é prova de melhora real** -- pode ser o modelo
  de referência se recalibrando ao patamar já ruim. Checar se o
  coeficiente de macro ou de idade mudou muito entre janelas antes de
  aceitar como "resolvido".
- **Dado sintético valida a lógica, não substitui a aplicação real.** A
  função `gerar_tabela_bruta_contratual` e a agregação Spark acima
  precisam ser substituídas pela leitura real da fonte truncada e pelo
  painel de fluxo por safra/MOB já existente (bloco m03b) + `df_macro`.
